Research code still deserves to be *correct, debuggable, and readable*. These practices cost little and repay themselves the first time a result looks wrong.

::: {.callout-tip}
## Why learn this?
Research code that silently returns a *wrong* answer can cost you months. Validation, tests and logging are cheap insurance that catches mistakes early — while they're still one line to fix.

- **Imagine you need** to catch a negative viscosity before it turns into NaNs three hours into a run — `raise` immediately with a clear message.
- **Imagine you need** to refactor your TKE routine without fear — a *test* tells you in one second whether you broke it.
:::

## Validation & exceptions
Fail early and clearly. Validate inputs and `raise` a specific exception with a helpful message rather than returning a silent wrong answer.

In [1]:
def reynolds(rho, u, length, mu):
    if mu <= 0:
        raise ValueError(f"viscosity must be > 0, got {mu}")
    return rho * u * length / mu

try:
    reynolds(1000, 1.0, 0.05, mu=0)
except ValueError as e:
    print('caught:', e)

print('ok:', reynolds(1000, 1.0, 0.05, mu=1e-3))

caught: viscosity must be > 0, got 0
ok: 50000.0


The full form is `try / except / else / finally`: `else` runs only if no exception occurred; `finally` always runs (cleanup). You can also define **custom exceptions** for your domain.

In [2]:
class ConvergenceError(RuntimeError):
    """Raised when an iterative solver fails to converge."""

def solve(max_iter=5):
    residual = 1.0
    for i in range(max_iter):
        residual *= 0.5
    if residual > 1e-6:
        raise ConvergenceError(f'stalled at residual={residual:.2e}')
    return residual

try:
    solve()
except ConvergenceError as e:
    print('solver problem:', e)
else:
    print('converged')
finally:
    print('cleanup always runs')

solver problem: stalled at residual=3.12e-02
cleanup always runs


**Imagine you need** to change your `normalise` routine and be *sure* you didn't break it. This is super easy in Python using a quick `assert` or a `unittest` case.

## Testing
Automated tests let you change code with confidence. `assert` is enough for quick checks; the standard-library `unittest` structures larger suites (the popular `pytest` is similar but a third-party install).

In [3]:
def normalise(xs):
    m = max(abs(x) for x in xs)
    return [x / m for x in xs]

# quick inline checks
assert normalise([2, -4]) == [0.5, -1.0]
assert abs(sum(normalise([1, 1, 1])) - 3.0) < 1e-12
print('inline asserts passed')

inline asserts passed


In [4]:
import unittest

class TestNormalise(unittest.TestCase):
    def test_scaling(self):
        self.assertEqual(normalise([2, -4]), [0.5, -1.0])
    def test_all_ones(self):
        self.assertAlmostEqual(max(normalise([1, 1, 1])), 1.0)

# run the suite inside the notebook
unittest.main(argv=['ignored', '-v'], exit=False)

test_all_ones (__main__.TestNormalise.test_all_ones) ... 

ok


test_scaling (__main__.TestNormalise.test_scaling) ... 

ok


----------------------------------------------------------------------
Ran 2 tests in 0.003s

OK


**Imagine you need** to find where a twelve-hour run stalled, without scattering `print` statements everywhere. This is super easy in Python using the `logging` module with timestamps and levels.

## Logging
Prefer `logging` over scattered `print` calls: you get severity **levels** (DEBUG/INFO/WARNING/ERROR), timestamps, and the ability to turn detail up or down without deleting code.

In [5]:
import logging, sys

logger = logging.getLogger('mech501.demo')
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter('%(levelname)s | %(name)s | %(message)s'))
logger.addHandler(handler)
logger.setLevel(logging.INFO)      # INFO and above are shown; DEBUG hidden

logger.debug('you will NOT see this (below the level)')
logger.info('starting run, Re=5000')
logger.warning('residual increased at step 12')

INFO | mech501.demo | starting run, Re=5000


WARNING | mech501.demo | residual increased at step 12


## Documentation
Write **docstrings** (what a thing does, its parameters and returns) and use **type hints** as living documentation. A common, tooling-friendly style is NumPy/Google format, which tools like *Sphinx* turn into web docs.

In [6]:
def darcy_friction(re: float) -> float:
    """Blasius correlation for turbulent pipe friction.

    Parameters
    ----------
    re : float
        Reynolds number (valid ~4e3 to 1e5).

    Returns
    -------
    float
        Darcy friction factor.
    """
    return 0.316 * re ** -0.25

help(darcy_friction)

Help on function darcy_friction in module __main__:

darcy_friction(re: float) -> float
    Blasius correlation for turbulent pipe friction.
    
    Parameters
    ----------
    re : float
        Reynolds number (valid ~4e3 to 1e5).
    
    Returns
    -------
    float
        Darcy friction factor.



## Self-tests

**1.** Write `safe_sqrt(x)` that raises `ValueError('negative input')` for `x < 0`, else returns `x ** 0.5`.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
def safe_sqrt(x):
    if x < 0:
        raise ValueError('negative input')
    return x ** 0.5

print(safe_sqrt(9))
try:
    safe_sqrt(-1)
except ValueError as e:
    print('caught:', e)

**2.** Add an `assert`-based test that `safe_sqrt(16) == 4`.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
assert safe_sqrt(16) == 4
print('passed')

## Turbulence in practice: a realizability check

A physically valid Reynolds-stress tensor must have non-negative normal stresses and satisfy the Cauchy--Schwarz bound $\langle u'v'\rangle^2 \le \langle u'^2\rangle\langle v'^2\rangle$. Validate it and raise on a violation (compare with Assignment 1, Q3).

In [9]:
def check_realizable(uu, vv, uv, tol=1e-12):
    """Raise ValueError if the Reynolds stresses are not realizable."""
    if uu < -tol or vv < -tol:
        raise ValueError('normal stresses must be non-negative')
    if uv ** 2 > uu * vv + tol:
        raise ValueError('Cauchy-Schwarz violated')
    return True

print('valid   :', check_realizable(1.0, 0.5, 0.4))
try:
    check_realizable(1.0, 0.5, 0.9)   # 0.81 > 0.5
except ValueError as e:
    print('rejected:', e)

valid   : True
rejected: Cauchy-Schwarz violated


**Self-test.** Write `check_tke(k)` that raises `ValueError` if the turbulent kinetic energy is negative, and returns `k` otherwise.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
def check_tke(k):
    if k < 0:
        raise ValueError('TKE must be non-negative')
    return k

print(check_tke(0.75))